In [1]:
import tensorflow as tf

device = tf.test.gpu_device_name()

if device == "/device:GPU:0":
  print("found cuda")
else:
  device = 'cpu'
  #raise SystemError('GPU device no found')

found cuda


I0000 00:00:1793521103.917493      48 gpu_device.cc:2020] Created device /device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793521103.920910      48 gpu_device.cc:2020] Created device /device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
import torch

device = torch.cuda.is_available()

if device:
  device = torch.device("cuda")
  print("found cuda")
else:
  device = "cpu"

#better version to write standard
device = 'cuda' if torch.cuda.is_available() else 'cpu'

found cuda


In [3]:
print(device)

cuda


In [4]:
!pip install wget

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=1651003016933d34ccec64138658f6e27d66637c602f668366d1759b25e54aa8
  Stored in directory: /root/.cache/pip/wheels/8a/b8/04/0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget


In [5]:
import wget
import os

url = 'https://nyu-mll.github.io/CoLA/cola_public_1.1.zip'

if not os.path.exists('./cola_public_1.1.zip'):
  wget.download(url, './cola_public_1.1.zip')

In [6]:
if not os.path.exists('./cola_pulic/'):
  !unzip cola_public_1.1.zip

Archive:  cola_public_1.1.zip
   creating: cola_public/
  inflating: cola_public/README      
   creating: cola_public/tokenized/
  inflating: cola_public/tokenized/in_domain_dev.tsv  
  inflating: cola_public/tokenized/in_domain_train.tsv  
  inflating: cola_public/tokenized/out_of_domain_dev.tsv  
   creating: cola_public/raw/
  inflating: cola_public/raw/in_domain_dev.tsv  
  inflating: cola_public/raw/in_domain_train.tsv  
  inflating: cola_public/raw/out_of_domain_dev.tsv  


In [7]:
import pandas as pd

df = pd.read_csv('cola_public/raw/in_domain_train.tsv',delimiter='\t', header=None,names=['sentence_source', 'label', 'label_notes', 'sentence'])

df.head()

,sentence_source,label,label_notes,sentence
0,gj04,1,NaN,"Our friends won't buy this analysis, let alone..."
1,gj04,1,NaN,One more pseudo generalization and I'm giving up.
2,gj04,1,NaN,One more pseudo generalization or I'm giving up.
3,gj04,1,NaN,"The more we study verbs, the crazier they get."
4,gj04,1,NaN,Day by day the facts are getting murkier.


In [8]:
df.loc[df.label == 0].sample(5)[['sentence','label']]

,sentence,label
4911,Every essay she's written and which I've read ...,0
7892,Bill's reading Shakespeare and Maureen's singi...,0
2150,Cats touch easily.,0
301,I finally worked up enough courage to ask whic...,0
7588,What Henry find was Bill sad.,0


In [9]:
#numpy values
sentence = df.sentence.values
label = df.label.values

In [10]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',do_lower_case=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
#our sentence
print("original", sentence[0])

#sentence split into tokens
print("tokenizes", tokenizer.tokenize(sentence[0]))

#printing ids
print("token ids",tokenizer.convert_tokens_to_ids(tokenizer.tokenize(sentence[0])))
#here is mistook printing tokens_to_ids to .ids which we do when we use endcoder

original Our friends won't buy this analysis, let alone the next one we propose.
tokenizes ['our', 'friends', 'won', "'", 't', 'buy', 'this', 'analysis', ',', 'let', 'alone', 'the', 'next', 'one', 'we', 'propose', '.']
token ids [2256, 2814, 2180, 1005, 1056, 4965, 2023, 4106, 1010, 2292, 2894, 1996, 2279, 2028, 2057, 16599, 1012]


In [12]:
max_len = 0

for sent in sentence:
  input_ids = tokenizer.encode(sent, add_special_tokens =True)

  max_len = max(max_len, len(input_ids))

print(max_len)

47


In [13]:
input_ids = []
attention_mask = []

for sent in sentence:
  encoded = tokenizer(
      sent,
      max_length = 256,
      return_tensors='pt',
      return_attention_mask = True,
      padding = 'max_length',
      truncation = True,
      add_special_tokens = True,
  )

  input_ids.append(encoded['input_ids'])
  attention_mask.append(encoded['attention_mask'])

input_ids = torch.cat(input_ids, dim=0)
attention_mask = torch.cat(attention_mask, dim=0)
labels = torch.tensor(label)

print('original: ', sentence[0])
print('Token IDs: ', input_ids[0])

original:  Our friends won't buy this analysis, let alone the next one we propose.
Token IDs:  tensor([  101,  2256,  2814,  2180,  1005,  1056,  4965,  2023,  4106,  1010,
         2292,  2894,  1996,  2279,  2028,  2057, 16599,  1012,   102,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     

In [14]:
from torch.utils.data import TensorDataset, random_split

dataset = TensorDataset(input_ids, attention_mask, labels)

train_size = int(0.8 * len(dataset))
val_size = int(len(dataset) - train_size)

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f'Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}')

Training samples: 6840, Validation samples: 1711


In [15]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

#we are using RandomSampler rather then doing shuffle = True
train_dataloader = DataLoader(train_dataset, batch_size=64, sampler = RandomSampler(train_dataset))

#using sequentialSampler rahtern then doing shuffle = False
val_dataloader = DataLoader(val_dataset, batch_size=64, sampler = SequentialSampler(val_dataset))

In [16]:
from transformers import BertForSequenceClassification, BertConfig

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2,
    output_attentions = False,
    output_hidden_states = False,
)

#writting this using cpu
if device == 'cuda':
  model.cuda()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

In [18]:
from transformers import get_linear_schedule_with_warmup

epochs = 10

total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps=0,
                                            num_training_steps = total_steps)

In [19]:
import numpy as np

def flat_accuracy(preds, labels):
  pred_flat = np.argmax(preds, axis=1).flatten()
  labels_flat = labels.flatten()
  return np.sum(pred_flat == labels_flat) / len(labels_flat)

In [20]:
import time
import datetime

def format_time(elapsed):
  elapsed_rounded = int(round(elapsed))

  return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
import random
import numpy as np

seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

training_stats = []
total_t0 = time.time()

for epoch in range(0, epochs):
  print(f"Epochs : {epoch + 1}, Total_epochs : {epochs}")

  t0 = time.time()

  total_train_loss = 0

  model.train()

  for step, batch in enumerate(train_dataloader):
    if step % 40 == 0 and step == 0:
      elapsed = format_time(time.time() - t0)
      print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)
    b_labels = batch[2].to(device)

    model.zero_grad()
    result = model(b_input_ids,
                  token_type_ids = None,
                  attention_mask = b_input_mask,
                  labels = b_labels,
                  return_dict = True)
      
    loss = result.loss
    logits = result.logits

    total_train_loss += loss.item()

    loss.backward()

    torch.nn.utils.clip_grad_norm(model.parameters(),1.0)

    optimizer.step()

    scheduler.step()

  avg_train_loss = total_train_loss / len(train_dataloader)

  training_time = format_time(time.time() - t0)


  print("")
  print("  Average training loss: {0:.2f}".format(avg_train_loss))
  print("  Training epcoh took: {:}".format(training_time))

  t0 = time.time()

  model.eval()

  total_eval_accuracy = 0
  total_eval_loss = 0
  nb_eval_steps = 0

  for batch in val_dataloader:

    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)
    b_labels = batch[2].to(device)

    with torch.no_grad():
      result = model(b_input_ids,
                     token_type_ids = None,
                     attention_mask = b_input_mask,
                     labels = b_labels,
                     return_dict= True)

    loss = result.loss
    logits = result.logits

    total_eval_loss += loss.item()
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    total_eval_accuracy += flat_accuracy(logits, label_ids)

  avg_val_accuracy = total_eval_accuracy / len(val_dataloader)
  avg_val_loss = total_eval_loss / len(val_dataloader) 

  validation_time = format_time(time.time() - t0)

  print("  Validation Loss: {0:.2f}".format(avg_val_loss))
  print("  Validation took: {:}".format(validation_time))

  # Record all statistics from this epoch.
  training_stats.append(
      {
          'epoch': epoch + 1,
          'Training Loss': avg_train_loss,
          'Valid. Loss': avg_val_loss,
          'Valid. Accur.': avg_val_accuracy,
          'Training Time': training_time,
          'Validation Time': validation_time
      }
  )

print("")
print("Training complete!")

print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

Epochs : 1, Total_epochs : 10
  Batch     0  of    107.    Elapsed: 0:00:00.


/tmp/ipykernel_48/1708192376.py:46: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(model.parameters(),1.0)



  Average training loss: 0.54
  Training epcoh took: 0:05:06
  Validation Loss: 0.46
  Validation took: 0:00:31
Epochs : 2, Total_epochs : 10
  Batch     0  of    107.    Elapsed: 0:00:00.

  Average training loss: 0.41
  Training epcoh took: 0:05:14
  Validation Loss: 0.45
  Validation took: 0:00:32
Epochs : 3, Total_epochs : 10
  Batch     0  of    107.    Elapsed: 0:00:00.

  Average training loss: 0.33
  Training epcoh took: 0:05:15
  Validation Loss: 0.48
  Validation took: 0:00:32
Epochs : 4, Total_epochs : 10
  Batch     0  of    107.    Elapsed: 0:00:00.

  Average training loss: 0.27
  Training epcoh took: 0:05:14
  Validation Loss: 0.46
  Validation took: 0:00:31
Epochs : 5, Total_epochs : 10
  Batch     0  of    107.    Elapsed: 0:00:00.

  Average training loss: 0.22
  Training epcoh took: 0:05:14


In [ ]:
import pandas as pd

pd.options.display.precision = 2

df_stats = pd.DataFrame(data=training_stats)

df_stats = df_stats.set_index('epoch')

df_stats

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='darkgrid')

sns.set(font_scale=1.5)
plt.rcParams["figure.figsize"] = (12,6)

plt.plot(df_stats['Training Loss'], 'b-o', label="Training")
plt.plot(df_stats['Valid. Loss'], 'g-o', label="Validation")

plt.title("Training & Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.xticks([1, 2, 3, 4])

plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("./cola_public/raw/out_of_domain_dev.tsv", delimiter='\t', header=None, names=['sentence_source', 'label', 'label_notes', 'sentence'])

print('Number of test sentences: {:,}\n'.format(df.shape[0]))

sentences = df.sentence.values
labels = df.label.values

input_ids = []
attention_masks = []

for sent in sentences:
    encoded_dict = tokenizer(
                          sent,
                          max_length = 256,
                          return_tensors='pt',
                          return_attention_mask = True,
                          padding = 'max_length',
                          truncation = True,
                          add_special_tokens = True,
                   )

    input_ids.append(encoded_dict['input_ids'])

    attention_masks.append(encoded_dict['attention_mask'])

input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

batch_size = 32

prediction_data = TensorDataset(input_ids, attention_masks, labels)
prediction_sampler = SequentialSampler(prediction_data)
prediction_dataloader = DataLoader(prediction_data, sampler=prediction_sampler, batch_size=batch_size)

In [ ]:
model.eval()

predictions , true_labels = [], []

for batch in prediction_dataloader:
  batch = tuple(t.to(device) for t in batch)

  b_input_ids, b_input_mask, b_labels = batch

  with torch.no_grad():
      result = model(b_input_ids,
                     token_type_ids=None,
                     attention_mask=b_input_mask,
                     return_dict=True)

  logits = result.logits

  logits = logits.detach().cpu().numpy()
  label_ids = b_labels.to('cpu').numpy()

  predictions.append(logits)
  true_labels.append(label_ids)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted Labels")
plt.ylabel("Actual Labels")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import os

output_dir = './model_save/'

if not os.path.exists(output_dir):
  os.makedirs(output_dir)

print(f"Saving model to {output_dir}")

model_to_save = mode.module if hasattr(model, 'module') else model
model_to_save.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)